In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
from functools import partial

import torch
import torch.nn as nn
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

from src.model import register_whisper_accent
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset
from src.train.train import processor_init, model_init, ModelArguments, LoraArguments
from src.train.trainer import compute_metrics
register_whisper_accent()

In [ ]:
model_args = ModelArguments()
lora_args = LoraArguments()

In [ ]:
processor = processor_init(model_args)
model = model_init(model_args, lora_args, processor)
model.print_trainable_parameters()


In [ ]:
collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
)

train_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    processor=processor,
    split="train",
    shuffle=True,
    num_proc=16,
)

eval_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    processor=processor,
    split="validation",
    shuffle=False,
    num_proc=16,
)

In [ ]:
import datetime

run_name = (
    f"whisper-accent-tiny-en-lora-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
)
output_dir = "/workspace/whisper-accent-tiny.en"

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    # max_steps=1000,
    max_steps=500,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    logging_steps=10,
    save_strategy="steps",
    # save_steps=200,
    save_steps=100,
    save_total_limit=100,
    bf16=True,
    fp16=False,
    eval_steps=100,
    run_name=run_name,
    optim="adamw_torch",
    # optim_args={"betas": (0.9, 0.999), "eps": 1e-8, "weight_decay": 0.01},
    report_to=["tensorboard"],
    push_to_hub=True,
    hub_model_id="mavleo96/whisper-accent-tiny.en",
    hub_strategy="all_checkpoints",
    gradient_checkpointing=False,
    predict_with_generate=True,
    remove_unused_columns=False,
)

In [ ]:
compute_metrics_fn = partial(compute_metrics, processor=processor)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    data_collator=collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
    compute_metrics=compute_metrics_fn,
)

In [ ]:
trainer.train()

In [ ]:
trainer.model.merge_and_unload().save_pretrained(f"{output_dir}")

In [ ]:
processor.save_pretrained(f"{output_dir}")

In [ ]:
trainer.push_to_hub()

In [ ]:
accent_embeddings = model.model.decoder.embed_tokens.weight[
    list(model.generation_config.accent_to_id.values()), :
]
accent_embeddings.shape

In [ ]:
torch.nn.functional.cosine_similarity(accent_embeddings, accent_embeddings, dim=1)